In [ ]:
import pandas as pd
import numpy as np
import os
import gzip
import json
import csv

os.chdir('../data/beauty')
os.getcwd()

In [ ]:
map_file = 'item_id_mapping.csv'
df = pd.read_csv(map_file)

map_dict = dict(zip(df['original_id'], df['mapped_id']))

print(f'shape: {df.shape}')
df.head()

In [ ]:
meta_path = 'meta.json.gz'

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

meta_df = getDF(meta_path)
meta_df.head()

In [ ]:
meta_df['item_id'] = meta_df['asin'].map(map_dict)
meta_df.dropna(subset=['item_id'], inplace=True)
meta_df['item_id'] = meta_df['item_id'].astype('int64')
meta_df.sort_values(by=['item_id'], inplace=True)

print(f'shape: {meta_df.shape}')
meta_df.head()

In [ ]:
ori_cols = meta_df.columns.tolist()
new_cols = [ori_cols[-1]] + ori_cols[:-1]

meta_df = meta_df[new_cols]

file = 'meta.csv'
meta_df.to_csv(file, index=False)

df = pd.read_csv(file)
df.head()

In [ ]:
print("Missing values in 'title':", df['title'].isnull().sum())
print("Missing values in 'description':", df['description'].isnull().sum())
print("Missing values in 'brand':", df['brand'].isnull().sum())
print("Missing values in 'categories':", df['categories'].isnull().sum())
print("Missing values in all four columns:", df[['title', 'description', 'brand', 'categories']].isnull().all(axis=1).sum())

df['description'] = df['description'].fillna(" ")
df['title'] = df['title'].fillna(" ")
df['brand'] = df['brand'].fillna(" ")
df['categories'] = df['categories'].fillna(" ")

In [ ]:
from sentence_transformers import SentenceTransformer
import array

os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"

In [ ]:
sentences = []
for i, row in df.iterrows():
    sen = row['title'] + ' ' + row['brand'] + ' '
    cates = eval(row['categories'])
    if isinstance(cates, list):
        for c in cates[0]:
            sen = sen + c + ' '
    sen += row['description']
    sen = sen.replace('\n', ' ')

    sentences.append(sen)

sentences[:5]

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_embeddings = model.encode(sentences)
sentence_embeddings[:5]

In [ ]:
np.save('text_feat.npy', sentence_embeddings)
text_feat = np.load('text_feat.npy', allow_pickle=True)
print(text_feat.shape)
text_feat[:5]

In [ ]:
def readImageFeatures(path):
  f = open(path, 'rb')
  while True:
    asin = f.read(10).decode('UTF-8')
    if asin == '': break
    a = array.array('f')
    a.fromfile(f, 4096)
    yield asin, a.tolist()
    
img_data = readImageFeatures("image_features.b")

In [ ]:
feats = {}
avg = []
for d in img_data:
    if d[0] in map_dict:
        feats[int(map_dict[d[0]])] = d[1]
        avg.append(d[1])
# avg = np.array(avg).mean(0).tolist()
batch_size = 10000
batch_avgs = []

for i in range(0, len(avg), batch_size):
    batch = avg[i:i + batch_size]

    batch_avg = np.array(batch).mean(0)
    batch_avgs.append(batch_avg)

    print(f"Processed {min(i + batch_size, len(avg))}/{len(avg)} records")

avg = np.array(batch_avgs).mean(0).tolist()

ret = []
non_no = []
for i in range(len(map_dict)):
    if i in feats:
        ret.append(feats[i])
    else:
        non_no.append(i)
        ret.append(avg)
ret = np.array(ret)
        
np.save('image_feat.npy', ret)
print('Missing number of image features:', len(non_no))
print(ret.shape)
ret[:5]